In [ ]:
# H-3 dev training, 2 x T4 version. Settings: Accelerator = GPU T4 x2, Internet off.
# Inputs: the 4 shared datasets. Then: Save Version -> Save & Run All (Commit).

In [ ]:
# Kaggle GPU runner (2 x T4): script 115, H-3 candidates (Amendment 9). One arm per GPU, run in parallel.
import glob, os, shutil, subprocess, sys, time
T0 = time.time()
ARMS = ["hybrid_mri_rank_enhw_mu_s0", "hybrid_mri_rank_mu_s1"]   # one arm per GPU
PACK = glob.glob("/kaggle/input/**/pg_inputs.npz", recursive=True)[0]
DS = os.path.dirname(PACK)
code = glob.glob("/kaggle/input/**/src/115_predictgbm_unet_dev.py", recursive=True)[0]
subprocess.run(["nvidia-smi"])
procs = []
for gpu, arm in enumerate(ARMS):
    repo = f"/kaggle/working/repo_{gpu}"
    shutil.copytree(os.path.dirname(os.path.dirname(code)), repo, dirs_exist_ok=True)
    env = dict(os.environ, GBM_PG_PACK=PACK, GBM_PG_TEST_IDS=os.path.join(DS, "test_ids.json"),
               GBM_MU_PACK=glob.glob("/kaggle/input/**/mu_train_2mm.npz", recursive=True)[0],
               GBM_PG_MRI=glob.glob("/kaggle/input/**/pg_mri2mm.npz", recursive=True)[0],
               GBM_DISK_CACHE=f"/kaggle/tmp/cache_{gpu}", CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    log = open(f"/kaggle/working/log_{arm}.txt", "w")
    procs.append((arm, repo, subprocess.Popen([sys.executable, "-u", "src/115_predictgbm_unet_dev.py", "--arms", arm,
                                                "--epochs", "60", "--tag", "_h3_" + arm], cwd=repo, env=env,
                                               stdout=log, stderr=subprocess.STDOUT)))
while any(p.poll() is None for _, _, p in procs):
    time.sleep(600)
    for arm, _, _ in procs:
        lines = open(f"/kaggle/working/log_{arm}.txt").read().splitlines()
        print(f"[{(time.time() - T0) / 3600:.1f} h] {arm}: {lines[-1] if lines else ''}", flush=True)
for arm, repo, p in procs:
    for f in glob.glob(os.path.join(repo, "output", "predictgbm", "*h3*")) + glob.glob(os.path.join(repo, "output", "predictgbm", "oof_logits_*.npz")):
        shutil.copy2(f, "/kaggle/working/")
    print(arm, "exit", p.returncode, flush=True)
print(f"done in {(time.time() - T0) / 3600:.2f} h", flush=True)
